# Predictive Benchmark Notebook

This notebook follows the required controlled benchmarking workflow and single-notebook protocol.

- File: `predictive_benchmark.ipynb`
- Reproducibility: fixed random seed and explicit execution order
- Constraint: later tasks must build on Task 1 assumptions unless explicitly challenged

In [1]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

print(f"Random seed fixed at {SEED}")

Random seed fixed at 42


## Task 1

### 1. Plan
1. Load the hotel booking dataset from the provided source files.
2. Audit structure, missingness, duplicates, and target balance.
3. Identify leakage-prone variables and define an admissible feature set.
4. Define reproducible preprocessing assumptions and a time-aware validation split.

### 2. Risks
- Data leakage: post-outcome fields (for example reservation final status fields) can inflate performance.
- Invalid evaluation: random split can leak temporal signal; use arrival-date-based split.
- Weak reproducibility: uncontrolled randomness or hidden preprocessing decisions.
- Misleading interpretation: confusing NA meaning (missing vs not applicable).
- Poor data quality: outliers, impossible values, duplicates.
- Unsupported assumptions: transformations without grounding in variable definitions.

### 3. Implementation

In [2]:
DATA_CANDIDATES = [
    Path('/Users/cindylow/Downloads/hotel_bookings (1).xlsx'),
    Path('/Users/cindylow/Downloads/hotel_bookings (1).csv'),
    Path('/Users/cindylow/Downloads/hotel_bookings.csv'),
]

selected_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if selected_path is None:
    raise FileNotFoundError('No dataset file found in expected locations.')

if selected_path.suffix.lower() == '.xlsx':
    df = pd.read_excel(selected_path)
else:
    df = pd.read_csv(selected_path)

print(f"Loaded: {selected_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
df.head(3)

Loaded: /Users/cindylow/Downloads/hotel_bookings (1).xlsx
Shape: (119390, 32)
Columns: 32


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


In [3]:
# Core data quality checks
summary = {
    'n_rows': len(df),
    'n_cols': df.shape[1],
    'duplicate_rows': int(df.duplicated().sum()),
    'target_exists': 'is_canceled' in df.columns,
}

missing_rate = (df.isna().mean().sort_values(ascending=False) * 100).round(2)

print(summary)
print('\nTop missing-rate columns (%)')
print(missing_rate.head(10))

{'n_rows': 119390, 'n_cols': 32, 'duplicate_rows': 31994, 'target_exists': True}

Top missing-rate columns (%)
company                   94.31
agent                     13.69
country                    0.41
children                   0.00
reserved_room_type         0.00
assigned_room_type         0.00
booking_changes            0.00
deposit_type               0.00
hotel                      0.00
previous_cancellations     0.00
dtype: float64


In [4]:
# Leakage risk controls based on variable semantics
TARGET = 'is_canceled'
POST_OUTCOME_COLUMNS = ['reservation_status', 'reservation_status_date']

assert TARGET in df.columns, 'Target column is missing.'

all_predictors = [c for c in df.columns if c != TARGET]
excluded_for_leakage = [c for c in POST_OUTCOME_COLUMNS if c in all_predictors]
feature_columns = [c for c in all_predictors if c not in excluded_for_leakage]

print('Target:', TARGET)
print('Excluded leakage-prone columns:', excluded_for_leakage)
print('Feature count after exclusion:', len(feature_columns))

Target: is_canceled
Excluded leakage-prone columns: ['reservation_status', 'reservation_status_date']
Feature count after exclusion: 29


In [5]:
# Build a time index from arrival fields to support temporal evaluation
arrival_date = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'].astype(str) + '-' +
    df['arrival_date_day_of_month'].astype(str),
    errors='coerce'
)

temporal_df = df.copy()
temporal_df['arrival_date'] = arrival_date

invalid_arrival_dates = temporal_df['arrival_date'].isna().sum()
print('Invalid arrival dates:', int(invalid_arrival_dates))

# Hold out 2017 arrivals as test to reduce temporal leakage from future into past
train_mask = temporal_df['arrival_date'] < pd.Timestamp('2017-01-01')
test_mask = temporal_df['arrival_date'] >= pd.Timestamp('2017-01-01')

print('Train rows:', int(train_mask.sum()))
print('Test rows :', int(test_mask.sum()))

print('\nTarget rate by split')
print(pd.DataFrame({
    'split': ['train', 'test'],
    'cancel_rate': [
        temporal_df.loc[train_mask, TARGET].mean(),
        temporal_df.loc[test_mask, TARGET].mean(),
    ]
}))

Invalid arrival dates: 0
Train rows: 78703
Test rows : 40687

Target rate by split
   split  cancel_rate
0  train     0.361854
1   test     0.386979


### 4. Verification
- Executability check: all cells run top-to-bottom with fixed seed and explicit file loading.
- Leakage check: `reservation_status` and `reservation_status_date` are excluded from predictors.
- Evaluation check: temporal split is used (`< 2017-01-01` train, `>= 2017-01-01` test).
- Data handling check: missingness is surfaced before modeling assumptions are applied.

In [6]:
# Single revision after verification:
# Issue detected: NA in 'agent' and 'company' is often semantic "not applicable" rather than random missingness.
# Revision: preserve this semantics with explicit category labels for reproducible preprocessing.

df_revised = temporal_df.copy()

for col in ['agent', 'company']:
    if col in df_revised.columns:
        df_revised[col] = df_revised[col].fillna('NOT_APPLICABLE').astype(str)

if 'country' in df_revised.columns:
    df_revised['country'] = df_revised['country'].fillna('UNKNOWN').astype(str)

if 'children' in df_revised.columns:
    df_revised['children'] = df_revised['children'].fillna(0)

print('Post-revision NA counts (selected columns):')
for c in ['agent', 'company', 'country', 'children']:
    if c in df_revised.columns:
        print(f"{c}: {df_revised[c].isna().sum()}")

Post-revision NA counts (selected columns):
agent: 0
company: 0
country: 0
children: 0


### 5. Revised final answer
- Dataset successfully loaded and audited.
- Leakage-prone variables excluded: `reservation_status`, `reservation_status_date`.
- Evaluation protocol established: temporal holdout split at **2017-01-01**.
- Preprocessing assumptions fixed for downstream tasks:
  - `agent`, `company`: missing treated as `NOT_APPLICABLE`
  - `country`: missing treated as `UNKNOWN`
  - `children`: missing treated as `0`

These assumptions form the baseline for Tasks 2–7 unless explicitly challenged later.

## Task 2

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 3

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 4

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 5

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 6

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.

## Task 7

### 1. Plan
Pending task-specific prompt.

### 2. Risks
- Data leakage
- Invalid evaluation
- Weak reproducibility
- Misleading interpretation
- Poor data quality
- Unsupported assumptions

### 3. Implementation
Pending task-specific prompt.

### 4. Verification
Pending task-specific prompt.

### 5. Revised final answer
Pending task-specific prompt.